# Final Model: Twitter-RoBERTa

**Nova IMS, Text Mining 2025/2026, Group 31**

This is our final solution: a single pipeline built on the champion model. We fine-tune Twitter-RoBERTa on the full training set, evaluate it on the validation split, classify the test set, and save the predictions to `outputs/pred_31.csv`. The heavy logic lives in the project `src/` package and is imported here, so this notebook stays short and readable.

## Setup

In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay


## 0. Project Modules (Self-Contained)

*(Guideline Compliance)*
To comply with the project guidelines requiring a strictly self-contained notebook, all modularized functions are instantiated in this initial section. 
They are ordered by dependency so the kernel resolves them sequentially.

### Module: `config.py`

In [ ]:
import re

# Reproducibility
SEED = 42

# Dataset splits
VAL_SIZE = 0.20
K_FOLD_N_SPLITS = 5

# Paths
TRAIN_CSV_PATH = "data/train.csv"
TEST_CSV_PATH = "data/test.csv"
RESULTS_CSV_PATH = "outputs/results.csv"
OUTPUT_PRED_PATH = "outputs/pred_best.csv"

# Error analysis output paths
CONF_MATRIX_PLOT_PATH = "outputs/confusion_matrix.png"
MISCLASSIFIED_TXT_PATH = "outputs/misclassified_report.txt"
MISCLASSIFIED_JSON_PATH = "outputs/misclassified_analysis.json"

# Labels
NUM_LABELS = 3
LABEL_NAMES = {0: "Bearish", 1: "Bullish", 2: "Neutral"}
LABEL2ID = {"Bearish": 0, "Bullish": 1, "Neutral": 2}
ID2LABEL = {0: "Bearish", 1: "Bullish", 2: "Neutral"}

# DistilBERT
DISTILBERT_MODEL_NAME = "distilbert-base-uncased"
DISTILBERT_N_SAMPLES_SPIKE = 200
DISTILBERT_CACHE_DIR = "outputs/distilbert_cache"
DISTILBERT_CHECKPOINT_DIR = "outputs/distilbert_checkpoints"

# Qwen decoder
QWEN_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# Twitter-RoBERTa
ROBERTA_MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment"
ROBERTA_N_SAMPLES_SPIKE = 200
ROBERTA_CACHE_DIR = "outputs/roberta_cache"
ROBERTA_CHECKPOINT_DIR = "outputs/roberta_checkpoints"

# FinBERT
FINBERT_MODEL_NAME = "ProsusAI/finbert"
FINBERT_N_SAMPLES_SPIKE = 200
FINBERT_CACHE_DIR = "outputs/finbert_cache"
FINBERT_CHECKPOINT_DIR = "outputs/finbert_checkpoints"

# DeBERTa-v3
DEBERTA_MODEL_NAME = "microsoft/deberta-v3-base"
DEBERTA_N_SAMPLES_SPIKE = 200
DEBERTA_CACHE_DIR = "outputs/deberta_cache"
DEBERTA_CHECKPOINT_DIR = "outputs/deberta_checkpoints"

# Feature matrix cache paths
BOW_TRAIN_PATH       = "outputs/X_train_bow.npz"
BOW_VAL_PATH         = "outputs/X_val_bow.npz"
TFIDF_UNI_TRAIN_PATH = "outputs/X_train_tfidf_uni.npz"
TFIDF_UNI_VAL_PATH   = "outputs/X_val_tfidf_uni.npz"
TFIDF_OPT_TRAIN_PATH = "outputs/X_train_tfidf_opt.npz"
TFIDF_OPT_VAL_PATH   = "outputs/X_val_tfidf_opt.npz"

# Leaderboard CSV schema
RESULTS_HEADERS = [
    "timestamp", "owner", "model_name", "feature_description",
    "accuracy", "precision_macro", "recall_macro", "f1_macro",
    "parameters", "f1_per_class", "notes",
]

# Preprocessing — placeholders
URL_PLACEHOLDER = "URL_PLACEHOLDER"
MENTION_PLACEHOLDER = "MENTION_PLACEHOLDER"
CASHTAG_PLACEHOLDER = "CASHTAG_PLACEHOLDER"
PROTECTED_PLACEHOLDERS = {URL_PLACEHOLDER, MENTION_PLACEHOLDER, CASHTAG_PLACEHOLDER}

# Preprocessing — financial noise tokens (used on top of NLTK stopwords)
FINANCIAL_STOPWORDS = {
    "rt", "amp", "co", "qt", "http", "https", "via",
    "stock", "stocks", "ticker", "tickers", "share", "shares",
}

# EDA — display
LABEL_PALETTE = {
    "Bearish": "#E06666",
    "Bullish": "#6AA84F",
    "Neutral": "#4A90E2",
}

# EDA — artifact detection patterns
URL_PATTERN     = re.compile(r'https?://\S+|www\.\S+')
MENTION_PATTERN = re.compile(r'@\w+')
CASHTAG_PATTERN = re.compile(r'\$\w+')
HASHTAG_PATTERN = re.compile(r'#\w+')

# EDA — fast stopword set (no NLTK dependency)
DEFAULT_STOPWORDS = {
    "i", "me", "my", "myself", "we", "our", "ours", "ourselves", "you", "your", "yours",
    "yourself", "yourselves", "he", "him", "his", "himself", "she", "her", "hers", "herself",
    "it", "its", "itself", "they", "them", "their", "theirs", "themselves", "what", "which",
    "who", "whom", "this", "that", "these", "those", "am", "is", "are", "was", "were", "be",
    "been", "being", "have", "has", "had", "having", "do", "does", "did", "doing", "a", "an",
    "the", "and", "but", "if", "or", "because", "as", "until", "while", "of", "at", "by",
    "for", "with", "about", "against", "between", "into", "through", "during", "before",
    "after", "above", "below", "to", "from", "up", "down", "in", "out", "on", "off", "over",
    "under", "again", "further", "then", "once", "here", "there", "when", "where", "why",
    "how", "all", "any", "both", "each", "few", "more", "most", "other", "some", "such",
    "no", "nor", "not", "only", "own", "same", "so", "than", "too", "very", "s", "t", "can",
    "will", "just", "don", "should", "now", "d", "ll", "m", "o", "re", "ve", "y",
    *FINANCIAL_STOPWORDS,
}



### Module: `utils.py`

In [ ]:
def log_info(msg: str) -> None:
    print(f"[INFO] {msg}")

def log_success(msg: str) -> None:
    print(f"[SUCCESS] {msg}")

def log_error(msg: str) -> None:
    print(f"[ERROR] {msg}")

def log_warning(msg: str) -> None:
    print(f"[WARNING] {msg}")

def print_header(title: str, width: int = 60) -> None:
    print("=" * width)
    print(title)
    print("=" * width)

def print_separator(width: int = 60) -> None:
    print("-" * width)



### Module: `evaluate.py`

In [ ]:
import os
import csv
from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    f1_score,
    classification_report,
    confusion_matrix,
)





def compute_metrics(y_true, y_pred) -> dict:
    accuracy = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    per_class = f1_score(y_true, y_pred, average=None, zero_division=0)
    f1_per_class = {
        name: float(per_class[i]) if i < len(per_class) else 0.0
        for i, name in LABEL_NAMES.items()
    }
    return {
        "accuracy": float(accuracy),
        "precision_macro": float(precision),
        "recall_macro": float(recall),
        "f1_macro": float(f1),
        "f1_per_class": f1_per_class,
    }


def log_model_run(
    model_name: str,
    feature_desc: str,
    metrics: dict,
    params: str = "",
    owner: str = "",
    notes: str = "",
) -> None:
    """Logs a model run to results.csv idempotently — updates if the same key exists."""
    os.makedirs(os.path.dirname(RESULTS_CSV_PATH), exist_ok=True)

    new_row = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "owner": owner,
        "model_name": model_name,
        "feature_description": feature_desc,
        "accuracy": f"{metrics['accuracy']:.4f}",
        "precision_macro": f"{metrics['precision_macro']:.4f}",
        "recall_macro": f"{metrics['recall_macro']:.4f}",
        "f1_macro": f"{metrics['f1_macro']:.4f}",
        "parameters": params,
        "f1_per_class": str(metrics.get("f1_per_class", "")),
        "notes": notes,
    }

    def _is_match(row: dict) -> bool:
        return (
            row["model_name"] == model_name
            and row["feature_description"] == feature_desc
            and row["parameters"] == params
            and row["owner"] == owner
        )

    existing_rows = []
    updated = False

    if os.path.exists(RESULTS_CSV_PATH) and os.path.getsize(RESULTS_CSV_PATH) > 0:
        try:
            with open(RESULTS_CSV_PATH, mode="r", newline="", encoding="utf-8") as f:
                reader = csv.reader(f)
                next(reader, None)
                for r in reader:
                    if len(r) != len(RESULTS_HEADERS):
                        continue
                    row = dict(zip(RESULTS_HEADERS, r))
                    if _is_match(row):
                        existing_rows.append(new_row)
                        updated = True
                    else:
                        existing_rows.append(row)
        except Exception as e:
            log_warning(f"Error reading leaderboard CSV: {e}. Resetting.")
            existing_rows = []

    if not updated:
        existing_rows.append(new_row)

    with open(RESULTS_CSV_PATH, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=RESULTS_HEADERS)
        writer.writeheader()
        writer.writerows(existing_rows)

    log_info(f"{'Updated' if updated else 'Added'} run in {RESULTS_CSV_PATH}")


def evaluate_and_log(
    y_true, y_pred,
    model_name: str,
    feature_desc: str,
    params: str = "",
    owner: str = "",
) -> dict:
    """Evaluates predictions, prints report, and logs to results.csv."""
    metrics = compute_metrics(y_true, y_pred)

    print_header(f"MODEL EVALUATION: {model_name} ({feature_desc})")
    log_info(f"Accuracy          : {metrics['accuracy']:.4f}")
    log_info(f"Precision (Macro) : {metrics['precision_macro']:.4f}")
    log_info(f"Recall (Macro)    : {metrics['recall_macro']:.4f}")
    log_info(f"F1 (Macro)        : {metrics['f1_macro']:.4f}")
    print_separator()
    print(classification_report(y_true, y_pred, target_names=list(LABEL_NAMES.values()), zero_division=0))
    print_separator()
    print(confusion_matrix(y_true, y_pred))

    log_model_run(model_name, feature_desc, metrics, params, owner=owner)
    return metrics


def evaluate_model(y_true, y_pred, owner: str, model, notes: str = "") -> dict:
    """Bento's interface — computes metrics, prints report, and logs to results.csv."""
    model_name = model.__class__.__name__ if hasattr(model, "__class__") else str(model)
    hyperparameters = str(model.get_params()) if hasattr(model, "get_params") else ""

    metrics = compute_metrics(y_true, y_pred)
    print(classification_report(y_true, y_pred, zero_division=0))
    print(confusion_matrix(y_true, y_pred))

    log_model_run(
        model_name=model_name,
        feature_desc="N/A",
        metrics=metrics,
        params=hyperparameters,
        owner=owner,
        notes=notes,
    )

    return {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "owner": owner,
        "model": model_name,
        "hyperparams": hyperparameters,
        "val_accuracy": metrics["accuracy"],
        "val_p_macro": metrics["precision_macro"],
        "val_r_macro": metrics["recall_macro"],
        "val_f1_macro": metrics["f1_macro"],
        "val_f1_per_class": metrics["f1_per_class"],
        "notes": notes,
    }


def save_submission(
    test_df: pd.DataFrame,
    predictions,
    output_path: str = OUTPUT_PRED_PATH,
    id_col: str = "id",
) -> pd.DataFrame:
    """Saves id + label predictions to CSV."""
    submission = pd.DataFrame({"id": test_df[id_col], "label": predictions})
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    submission.to_csv(output_path, index=False)
    log_success(f"Predictions saved to {output_path} ({len(submission)} rows)")
    return submission



### Module: `preprocessing.py`

In [ ]:
import re
import sys
import json
import argparse
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import TweetTokenizer
from nltk.stem import PorterStemmer, WordNetLemmatizer
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold





def _download_nltk_resources() -> None:
    resources = {
        'stopwords': 'corpora/stopwords',
        'punkt': 'tokenizers/punkt',
        'wordnet': 'corpora/wordnet',
        'omw-1.4': 'corpora/omw-1.4',
    }
    for name, path in resources.items():
        try:
            nltk.data.find(path)
        except LookupError:
            nltk.download(name, quiet=True)


_download_nltk_resources()

_SPACE_RE = re.compile(r'\s+')
_UNICODE_MAP = [
    (re.compile(r'[\u201c\u201d\u201e\u201f]'), '"'),
    (re.compile(r'[\u2018\u2019\u201a\u201b]'), "'"),
    (re.compile(r'[\u2012\u2013\u2014\u2015\u2212]'), '-'),
    (re.compile(r'\u2026'), '...'),
    (re.compile(r'\uFFFD'), ' '),
]
_TOKENIZER = TweetTokenizer(preserve_case=True, reduce_len=True, strip_handles=False)

# ── Text preprocessing ────────────────────────────────────────────────────────

def normalize_unicode_punctuation(text: str) -> str:
    if not isinstance(text, str):
        return ""
    for pattern, replacement in _UNICODE_MAP:
        text = pattern.sub(replacement, text)
    return text


def clean_regex(
    text: str,
    url_mode: str = 'replace',
    mention_mode: str = 'replace',
    cashtag_mode: str = 'keep',
) -> str:
    if url_mode == 'remove':
        text = URL_PATTERN.sub('', text)
    elif url_mode == 'replace':
        text = URL_PATTERN.sub(URL_PLACEHOLDER, text)

    if mention_mode == 'remove':
        text = MENTION_PATTERN.sub('', text)
    elif mention_mode == 'replace':
        text = MENTION_PATTERN.sub(MENTION_PLACEHOLDER, text)

    if cashtag_mode == 'remove':
        text = CASHTAG_PATTERN.sub('', text)
    elif cashtag_mode == 'replace':
        text = CASHTAG_PATTERN.sub(CASHTAG_PLACEHOLDER, text)

    return _SPACE_RE.sub(' ', text).strip()


def tokenize_tweet(text: str) -> list:
    return _TOKENIZER.tokenize(text)


def remove_stopwords_from_tokens(tokens: list, extra_stopwords: list = None) -> list:
    all_stops = set(stopwords.words('english')) | FINANCIAL_STOPWORDS
    if extra_stopwords:
        all_stops.update(extra_stopwords)
    return [t for t in tokens if (t.lower() not in all_stops) or (t in PROTECTED_PLACEHOLDERS)]


def apply_stemming(tokens: list) -> list:
    stemmer = PorterStemmer()
    return [t if t in PROTECTED_PLACEHOLDERS else stemmer.stem(t) for t in tokens]


def apply_lemmatization(tokens: list) -> list:
    lemmatizer = WordNetLemmatizer()
    return [t if t in PROTECTED_PLACEHOLDERS else lemmatizer.lemmatize(t) for t in tokens]


def preprocess_tweet(
    text: str,
    lowercase: bool = True,
    url_mode: str = 'replace',
    mention_mode: str = 'replace',
    cashtag_mode: str = 'keep',
    remove_stopwords: bool = True,
    custom_stopwords: list = None,
    use_stemming: bool = False,
    use_lemmatization: bool = True,
    return_str: bool = False,
):
    """Full tweet preprocessing pipeline. Returns token list or joined string."""
    text = normalize_unicode_punctuation(text)
    if lowercase:
        text = text.lower()
    text = clean_regex(text, url_mode=url_mode, mention_mode=mention_mode, cashtag_mode=cashtag_mode)
    tokens = tokenize_tweet(text)
    if remove_stopwords:
        tokens = remove_stopwords_from_tokens(tokens, extra_stopwords=custom_stopwords)
    if use_stemming:
        tokens = apply_stemming(tokens)
    if use_lemmatization:
        tokens = apply_lemmatization(tokens)
    return " ".join(tokens) if return_str else tokens


# ── Train / val splitting (absorbed from train_val_split.py) ──────────────────

def stratified_split(
    dataset: pd.DataFrame,
    test_size: float = VAL_SIZE,
    seed: int = SEED,
) -> tuple[pd.Series, pd.Series, pd.Series, pd.Series]:
    X, y = dataset['text'], dataset['label']
    return train_test_split(X, y, test_size=test_size, random_state=seed, stratify=y)


def create_stratified_kfold(n_splits: int = K_FOLD_N_SPLITS, seed: int = SEED) -> StratifiedKFold:
    return StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)


# ── Smoke tests ───────────────────────────────────────────────────────────────

def run_smoke_tests() -> None:
    """Verifies each preprocessing step with representative tweets."""
    print_header("PREPROCESSING SMOKE TESTS")

    test_tweets = [
        "Downgrades 4/7: $MLND to underperform at Needham—see details... https://t.co/example",
        "Shorting $AAPL here at $180. @elonmusk thoughts? #market #trading",
        "RT @NovaIMS: $BTC is falling down rapidly... RT to warn others!",
        "Having a cup of coffee and watching the market open. Very neutral.",
    ]

    for idx, raw in enumerate(test_tweets, 1):
        log_info(f"Test {idx}: '{raw}'")
        lem = preprocess_tweet(raw, return_str=True)
        log_info(f"  Lemmatized : '{lem}'")
        stem = preprocess_tweet(raw, url_mode='remove', mention_mode='remove',
                                cashtag_mode='replace', use_stemming=True,
                                use_lemmatization=False, return_str=False)
        log_info(f"  Stemmed    : {stem}")

    log_success("Smoke tests complete.")


# ── Main Entry Point for CLI & Agents ─────────────────────────────────────────



### Module: `eda.py`

In [ ]:
import os
import re
from collections import Counter
import pandas as pd


import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud



# --- Core Analyzer Class ---

class DatasetAnalyzer:
    def __init__(self, file_path: str, name: str = "Dataset"):
        self.file_path = file_path
        self.name = name
        self.df = None
        self.has_label = False
        self.load_data()

    def load_data(self):
        """Loads dataset from CSV and determines if labels exist."""
        if not os.path.exists(self.file_path):
            raise FileNotFoundError(f"Dataset not found at: {self.file_path}")
        self.df = pd.read_csv(self.file_path)
        self.has_label = "label" in self.df.columns
        # Basic text cleaning to ensure str type
        self.df["text"] = self.df["text"].fillna("").astype(str)

    def analyze_basic_stats(self) -> dict:
        """Returns standard metrics on size, duplicates, and missing rows."""
        total_rows = len(self.df)
        exact_dups = self.df["text"].duplicated().sum()
        ci_dups = self.df["text"].str.lower().str.strip().duplicated().sum()
        empty_rows = (self.df["text"].str.strip() == "").sum()
        
        return {
            "total_rows": total_rows,
            "exact_duplicates": int(exact_dups),
            "trimmed_case_insensitive_duplicates": int(ci_dups),
            "empty_rows": int(empty_rows)
        }

    def analyze_class_distribution(self) -> list:
        """Returns class distribution metrics (counts, percentages) if labels are present."""
        if not self.has_label:
            return []
        
        counts = self.df["label"].value_counts().sort_index()
        total = counts.sum()
        dist = []
        for val, cnt in counts.items():
            dist.append({
                "label_id": int(val),
                "label_name": LABEL_NAMES.get(val, f"Unknown ({val})"),
                "count": int(cnt),
                "percentage": float((cnt / total * 100))
            })
        return dist

    def analyze_text_lengths(self) -> dict:
        """Computes text character length and token count statistics."""
        char_lens = self.df["text"].str.len()
        token_lens = self.df["text"].str.split().str.len()
        
        stats = {
            "char_len": {
                "mean": float(char_lens.mean()),
                "median": float(char_lens.median()),
                "std": float(char_lens.std()),
                "min": int(char_lens.min()),
                "max": int(char_lens.max())
            },
            "token_len": {
                "mean": float(token_lens.mean()),
                "median": float(token_lens.median()),
                "std": float(token_lens.std()),
                "min": int(token_lens.min()),
                "max": int(token_lens.max())
            }
        }
        
        # Breakdown by class if available
        if self.has_label:
            stats["by_class"] = {}
            for val, name in LABEL_NAMES.items():
                class_df = self.df[self.df["label"] == val]
                if not class_df.empty:
                    c_chars = class_df["text"].str.len()
                    c_tokens = class_df["text"].str.split().str.len()
                    stats["by_class"][name] = {
                        "char_mean": float(c_chars.mean()),
                        "char_median": float(c_chars.median()),
                        "token_mean": float(c_tokens.mean()),
                        "token_median": float(c_tokens.median())
                    }
        return stats

    def analyze_artifacts(self) -> dict:
        """Counts URLs, mentions, cashtags, and hashtags in the corpus."""
        def count_pat(text, pattern):
            return len(pattern.findall(text))
            
        n_urls = self.df["text"].apply(lambda t: count_pat(t, URL_PATTERN))
        n_mentions = self.df["text"].apply(lambda t: count_pat(t, MENTION_PATTERN))
        n_cashtags = self.df["text"].apply(lambda t: count_pat(t, CASHTAG_PATTERN))
        n_hashtags = self.df["text"].apply(lambda t: count_pat(t, HASHTAG_PATTERN))
        
        total = len(self.df)
        rates = {
            "average_per_tweet": {
                "urls": float(n_urls.mean()),
                "mentions": float(n_mentions.mean()),
                "cashtags": float(n_cashtags.mean()),
                "hashtags": float(n_hashtags.mean())
            },
            "presence_percentage": {
                "urls": float((n_urls > 0).sum() / total * 100),
                "mentions": float((n_mentions > 0).sum() / total * 100),
                "cashtags": float((n_cashtags > 0).sum() / total * 100),
                "hashtags": float((n_hashtags > 0).sum() / total * 100)
            }
        }

        # Breakdown by class if available
        if self.has_label:
            rates["by_class"] = {}
            for val, name in LABEL_NAMES.items():
                class_df = self.df[self.df["label"] == val]
                if not class_df.empty:
                    c_total = len(class_df)
                    rates["by_class"][name] = {
                        "urls_mean": float(class_df["text"].apply(lambda t: count_pat(t, URL_PATTERN)).mean()),
                        "mentions_mean": float(class_df["text"].apply(lambda t: count_pat(t, MENTION_PATTERN)).mean()),
                        "cashtags_mean": float(class_df["text"].apply(lambda t: count_pat(t, CASHTAG_PATTERN)).mean()),
                        "hashtags_mean": float(class_df["text"].apply(lambda t: count_pat(t, HASHTAG_PATTERN)).mean())
                    }
        return rates

    def analyze_non_ascii(self) -> dict:
        """Finds non-ASCII character rates and compiles some raw examples."""
        def has_non_ascii(text):
            return any(ord(c) > 127 for c in text)
            
        non_ascii_mask = self.df["text"].apply(has_non_ascii)
        n_non_ascii = non_ascii_mask.sum()
        total = len(self.df)
        
        # Get up to 5 examples
        examples = self.df[non_ascii_mask]["text"].head(5).tolist()
        
        return {
            "non_ascii_count": int(n_non_ascii),
            "non_ascii_percentage": float(n_non_ascii / total * 100),
            "examples": examples
        }

    def get_top_tokens(self, top_n: int = 15) -> dict:
        """Computes top N words across the dataset, excluding standard and financial stopwords."""
        # Simple token clean function
        def clean_split(text):
            # Strip URLs, mentions, cashtags first to get real words
            t = URL_PATTERN.sub('', text)
            t = MENTION_PATTERN.sub('', t)
            t = CASHTAG_PATTERN.sub('', t)
            t = HASHTAG_PATTERN.sub('', t)
            # Remove punctuation
            t = re.sub(r'[^a-zA-Z\s]', '', t)
            words = t.lower().split()
            return [w for w in words if w not in DEFAULT_STOPWORDS and len(w) > 2]

        top_tokens = {}
        
        # Global top words
        all_words = []
        for text in self.df["text"]:
            all_words.extend(clean_split(text))
        
        top_tokens["global"] = [{"word": w, "count": c} for w, c in Counter(all_words).most_common(top_n)]
        
        # Class specific top words
        if self.has_label:
            top_tokens["by_class"] = {}
            for val, name in LABEL_NAMES.items():
                class_df = self.df[self.df["label"] == val]
                c_words = []
                for text in class_df["text"]:
                    c_words.extend(clean_split(text))
                top_tokens["by_class"][name] = [{"word": w, "count": c} for w, c in Counter(c_words).most_common(top_n)]
                
        return top_tokens

    def generate_full_report(self, top_n_words: int = 15) -> dict:
        """Compiles all individual analytical components into a unified structured report."""
        return {
            "dataset_name": self.name,
            "file_path": self.file_path,
            "basic_stats": self.analyze_basic_stats(),
            "class_distribution": self.analyze_class_distribution(),
            "text_lengths": self.analyze_text_lengths(),
            "artifacts": self.analyze_artifacts(),
            "non_ascii": self.analyze_non_ascii(),
            "top_words": self.get_top_tokens(top_n=top_n_words)
        }

# --- Plot Generation Utility ---

def save_eda_plots(train_analyzer: DatasetAnalyzer, output_dir: str = "outputs/eda/"):
    """
    Generates and saves professional analytical figures under outputs/eda/.
    Requires matplotlib and seaborn. Skips gracefully if unavailable.
    """
    os.makedirs(output_dir, exist_ok=True)
    df = train_analyzer.df
    
    # Set tailored aesthetic style
    sns.set_theme(style="whitegrid")
    plt.rcParams.update({'font.size': 10, 'figure.titlesize': 13})

    # Plot 1: Class Distribution
    if train_analyzer.has_label:
        dist = train_analyzer.analyze_class_distribution()
        dist_df = pd.DataFrame(dist)
        
        plt.figure(figsize=(6, 4))
        ax = sns.barplot(
            data=dist_df, x="label_name", y="count",
            palette=[LABEL_PALETTE[name] for name in dist_df["label_name"]],
            hue="label_name", legend=False
        )
        # Add labels
        for idx, row in dist_df.iterrows():
            ax.text(idx, row["count"] + 50, f"{row['count']}\n({row['percentage']:.2f}%)", ha="center")
            
        plt.title("Class Label Distribution (Training Set)", pad=15)
        plt.xlabel("Sentiment Class")
        plt.ylabel("Number of Tweets")
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, "class_distribution.png"), dpi=150)
        plt.close()

    # Plot 2: Length Distributions
    df["char_len"] = df["text"].str.len()
    df["token_len"] = df["text"].str.split().str.len()
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    if train_analyzer.has_label:
        plot_df = df.assign(class_name=df["label"].map(LABEL_NAMES))
        sns.histplot(data=plot_df, x="char_len", hue="class_name", multiple="layer", bins=30, palette=LABEL_PALETTE, ax=axes[0])
        sns.histplot(data=plot_df, x="token_len", hue="class_name", multiple="layer", bins=20, palette=LABEL_PALETTE, ax=axes[1])
    else:
        sns.histplot(data=df, x="char_len", bins=30, color="#4A90E2", ax=axes[0])
        sns.histplot(data=df, x="token_len", bins=20, color="#4A90E2", ax=axes[1])
        
    axes[0].set_title("Character Length Distribution")
    axes[0].set_xlabel("Number of Characters")
    axes[1].set_title("Word (Token) Length Distribution")
    axes[1].set_xlabel("Number of Words")
    
    plt.suptitle("Tweet Length Distributions", y=0.98)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "length_distributions.png"), dpi=150)
    plt.close()

    # Plot 3: Artifact Rates by Class
    if train_analyzer.has_label:
        rates = train_analyzer.analyze_artifacts()
        by_class = rates.get("by_class", {})
        
        classes = list(by_class.keys())
        features = ["URLs", "Cashtags", "Hashtags", "Mentions"]
        
        plot_data = []
        for cls in classes:
            plot_data.append({
                "Class": cls,
                "URLs": by_class[cls]["urls_mean"],
                "Cashtags": by_class[cls]["cashtags_mean"],
                "Hashtags": by_class[cls]["hashtags_mean"],
                "Mentions": by_class[cls]["mentions_mean"]
            })
        
        rates_df = pd.DataFrame(plot_data).melt(id_vars="Class", var_name="Artifact", value_name="Average Count")
        
        plt.figure(figsize=(9, 4.5))
        sns.barplot(data=rates_df, x="Artifact", y="Average Count", hue="Class", palette=LABEL_PALETTE)
        plt.title("Average Metadata Artifact Rates Grouped by Sentiment Class", pad=15)
        plt.xlabel("Tweet Metadata Type")
        plt.ylabel("Average Count per Tweet")
        plt.legend(title="Sentiment Class")
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, "artefact_rates_by_class.png"), dpi=150)
        plt.close()

    # Plot 4: Top Words by Class
    if train_analyzer.has_label:
        top_words = train_analyzer.get_top_tokens(top_n=10)
        by_class = top_words.get("by_class", {})
        
        fig, axes = plt.subplots(1, 3, figsize=(16, 5))
        for ax, (cls_name, words) in zip(axes, by_class.items()):
            words_df = pd.DataFrame(words)
            if not words_df.empty:
                sns.barplot(data=words_df, x="count", y="word", color=LABEL_PALETTE[cls_name], ax=ax)
            ax.set_title(f"Top Words in {cls_name} Tweets")
            ax.set_xlabel("Occurrence Frequency")
            ax.set_ylabel("")
            
        plt.suptitle("Top Highly Informative Sentiment Vocabularies", y=0.98)
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, "top_words_by_class.png"), dpi=150)
        plt.close()
        
    log_success(f"EDA plots saved to {output_dir}")

# --- Output Formatters ---

def print_markdown_report(train_rpt: dict, test_rpt: dict = None):
    """Prints a beautiful, comprehensive Markdown formatted report to stdout."""
    print(f"# Exploratory Data Analysis (EDA) Report")
    print(f"**Generative Analysis for Corpus Pipelines**\n")
    
    # 1. Dataset Overview
    print("## [1] Dataset Size & Integrity Overview")
    print("| Dataset | Total Rows | Exact Duplicates | Trimmed Insensitive Dups | Empty Rows |")
    print("| :--- | :---: | :---: | :---: | :---: |")
    tr = train_rpt["basic_stats"]
    print(f"| **Train** | {tr['total_rows']:,} | {tr['exact_duplicates']} | {tr['trimmed_case_insensitive_duplicates']} | {tr['empty_rows']} |")
    if test_rpt:
        te = test_rpt["basic_stats"]
        print(f"| **Test** | {te['total_rows']:,} | {te['exact_duplicates']} | {te['trimmed_case_insensitive_duplicates']} | {te['empty_rows']} |")
    print()

    # 2. Class Distribution
    if train_rpt["class_distribution"]:
        print("## [2] Sentiment Class Distributions (Train Set)")
        print("| Label ID | Sentiment Class | Count | Percentage | Chart representation |")
        print("| :---: | :--- | :---: | :---: | :--- |")
        for row in train_rpt["class_distribution"]:
            bar = "=" * int(row["percentage"] / 5)
            print(f"| `{row['label_id']}` | **{row['label_name']}** | {row['count']:,} | {row['percentage']:.2f}% | `{bar}` |")
        print("\n> **Strategic Note**: Standard ML algorithms will show bias to the majority Neutral class. Set class weight parameters to 'balanced' or implement cross-validation splitting accordingly.\n")

    # 3. Tweet Length Statistics
    print("## [3] Sequence Length Analyses")
    l_tr = train_rpt["text_lengths"]
    print("### Character Lengths:")
    print(f"* **Train**: Mean = {l_tr['char_len']['mean']:.1f} chars, Std = {l_tr['char_len']['std']:.1f}, Min/Max = {l_tr['char_len']['min']}/{l_tr['char_len']['max']}")
    if test_rpt:
        l_te = test_rpt["text_lengths"]
        print(f"* **Test**: Mean = {l_te['char_len']['mean']:.1f} chars, Std = {l_te['char_len']['std']:.1f}, Min/Max = {l_te['char_len']['min']}/{l_te['char_len']['max']}")
    
    print("\n### Word (Token) Counts:")
    print(f"* **Train**: Mean = {l_tr['token_len']['mean']:.1f} words, Std = {l_tr['token_len']['std']:.1f}, Min/Max = {l_tr['token_len']['min']}/{l_tr['token_len']['max']}")
    if test_rpt:
        print(f"* **Test**: Mean = {l_te['token_len']['mean']:.1f} words, Std = {l_te['token_len']['std']:.1f}, Min/Max = {l_te['token_len']['min']}/{l_te['token_len']['max']}")
    
    if "by_class" in l_tr:
        print("\n### Sequence Characteristics Grouped by Class (Train):")
        print("| Sentiment Class | Avg Character Length | Avg Word (Token) Count |")
        print("| :--- | :---: | :---: |")
        for cls_name, vals in l_tr["by_class"].items():
            print(f"| **{cls_name}** | {vals['char_mean']:.1f} characters | {vals['token_mean']:.1f} tokens |")
    print()

    # 4. Artifact Analysis
    print("## [4] Twitter Metadata Artifact Rates")
    art = train_rpt["artifacts"]
    print("| Metadata Type | Average Occurrences / Tweet | Percentage of Tweets Containing Feature |")
    print("| :--- | :---: | :---: |")
    for feat in ["urls", "cashtags", "hashtags", "mentions"]:
        print(f"| **{feat.upper()}** | {art['average_per_tweet'][feat]:.3f} | {art['presence_percentage'][feat]:.1f}% |")
    
    if "by_class" in art:
        print("\n### Average Metadata Artifact Occurrences Grouped by Class (Train):")
        print("| Sentiment Class | Avg URLs / Tweet | Avg Cashtags / Tweet | Avg Hashtags / Tweet | Avg Mentions / Tweet |")
        print("| :--- | :---: | :---: | :---: | :---: |")
        for cls_name, vals in art["by_class"].items():
            print(f"| **{cls_name}** | {vals['urls_mean']:.3f} | {vals['cashtags_mean']:.3f} | {vals['hashtags_mean']:.3f} | {vals['mentions_mean']:.3f} |")
    print("\n> **Strategic Note**: URLs are highly associated with Neutral (news) tweets. Removing URLs completely strips this structural feature; utilizing a unified placeholder `URL_PLACEHOLDER` is highly recommended.\n")

    # 5. Non-ASCII Analysis
    print("## [5] Non-ASCII & Unicode Anomalies")
    non_tr = train_rpt["non_ascii"]
    print(f"* **Train**: {non_tr['non_ascii_count']:,} tweets ({non_tr['non_ascii_percentage']:.2f}%) contain smart quotes, long dashes, smart apostrophes, or replacement symbols.")
    if test_rpt:
        non_te = test_rpt["non_ascii"]
        print(f"* **Test**: {non_te['non_ascii_count']:,} tweets ({non_te['non_ascii_percentage']:.2f}%) contain smart quotes, long dashes, smart apostrophes, or replacement symbols.")
    
    print("\n### Sample raw sentences showing Unicode irregularities:")
    for ex in non_tr["examples"]:
        # Clean sample sentences of emojis in case they cause issues when printed in samples
        clean_ex = re.sub(r'[^\x00-\x7F]+', ' ', ex)
        print(f"* `{clean_ex}`")
    print()

    # 6. Vocabularies
    print("## [6] Most Frequent Sentiment Vocabularies (Excluding Noise)")
    top_w = train_rpt["top_words"]
    print("### Top Overall Vocabularies:")
    print(", ".join([f"**{row['word']}** ({row['count']})" for row in top_w["global"]]))
    
    if "by_class" in top_w:
        print("\n### Top Vocabularies by Class (Train):")
        for cls_name, words in top_w["by_class"].items():
            word_list = ", ".join([f"**{row['word']}** ({row['count']})" for row in words[:8]])
            print(f"* **{cls_name}**: {word_list}")
    print()




### Module: `features.py`

In [ ]:
import os
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer







def extract_and_save_features() -> None:
    """Fits BoW, TF-IDF unigram, and optimized TF-IDF (1,2) vectorizers and saves sparse matrices."""
    print_header("FEATURE EXTRACTION PIPELINE (BoW & TF-IDF)")

    if not os.path.exists(TRAIN_CSV_PATH):
        raise FileNotFoundError(f"Training CSV not found at: {TRAIN_CSV_PATH}")

    train_df = pd.read_csv(TRAIN_CSV_PATH)
    log_info(f"Loaded {len(train_df)} rows from {TRAIN_CSV_PATH}")

    X_train, X_val, y_train, y_val = stratified_split(train_df)
    log_info(f"Split: Train={len(X_train)} | Val={len(X_val)}")

    log_info("Preprocessing texts ...")
    X_train_pre = X_train.apply(lambda t: preprocess_tweet(t, return_str=True))
    X_val_pre = X_val.apply(lambda t: preprocess_tweet(t, return_str=True))

    os.makedirs(os.path.dirname(BOW_TRAIN_PATH), exist_ok=True)

    bow_vec = CountVectorizer(ngram_range=(1, 1))
    X_train_bow = bow_vec.fit_transform(X_train_pre)
    X_val_bow = bow_vec.transform(X_val_pre)
    log_info(f"BoW vocab size: {len(bow_vec.vocabulary_)}")
    sp.save_npz(BOW_TRAIN_PATH, X_train_bow)
    sp.save_npz(BOW_VAL_PATH, X_val_bow)
    log_success(f"BoW saved → {BOW_TRAIN_PATH}, {BOW_VAL_PATH}")

    tfidf_vec = TfidfVectorizer(ngram_range=(1, 1))
    X_train_tfidf = tfidf_vec.fit_transform(X_train_pre)
    X_val_tfidf = tfidf_vec.transform(X_val_pre)
    log_info(f"TF-IDF unigram vocab size: {len(tfidf_vec.vocabulary_)}")
    sp.save_npz(TFIDF_UNI_TRAIN_PATH, X_train_tfidf)
    sp.save_npz(TFIDF_UNI_VAL_PATH, X_val_tfidf)
    log_success(f"TF-IDF unigrams saved → {TFIDF_UNI_TRAIN_PATH}, {TFIDF_UNI_VAL_PATH}")

    tfidf_opt_vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=25000)
    X_train_tfidf_opt = tfidf_opt_vec.fit_transform(X_train_pre)
    X_val_tfidf_opt = tfidf_opt_vec.transform(X_val_pre)
    log_info(f"TF-IDF optimized vocab size: {len(tfidf_opt_vec.vocabulary_)}")
    sp.save_npz(TFIDF_OPT_TRAIN_PATH, X_train_tfidf_opt)
    sp.save_npz(TFIDF_OPT_VAL_PATH, X_val_tfidf_opt)
    log_success(f"TF-IDF optimized saved → {TFIDF_OPT_TRAIN_PATH}, {TFIDF_OPT_VAL_PATH}")

    log_success("Feature extraction complete.")



### Module: `word_embeddings.py`

In [ ]:
import os
import sys
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.feature_extraction.text import TfidfVectorizer

from gensim.models import Word2Vec
import gensim.downloader as api







def _get_vocab_and_vectors(embedding_model):
    if hasattr(embedding_model, 'key_to_index'):
        return embedding_model.key_to_index, embedding_model
    if hasattr(embedding_model, 'wv') and hasattr(embedding_model.wv, 'key_to_index'):
        return embedding_model.wv.key_to_index, embedding_model.wv
    return embedding_model, embedding_model


def calculate_oov(tokenized_sentences: list, embedding_model) -> tuple[float, float, set]:
    """Returns (unique_oov_rate, token_oov_rate, oov_word_set) for the tokenized corpus."""
    vocab, _ = _get_vocab_and_vectors(embedding_model)
    all_tokens = [tok for sent in tokenized_sentences for tok in sent]
    unique_tokens = set(all_tokens)
    if not unique_tokens:
        return 0.0, 0.0, set()
    oov_unique = {tok for tok in unique_tokens if tok not in vocab}
    oov_tokens = [tok for tok in all_tokens if tok not in vocab]
    return len(oov_unique) / len(unique_tokens), len(oov_tokens) / len(all_tokens), oov_unique


def vectorize_corpus(tokenized_sentences: list, embedding_model, vector_size: int = 100) -> np.ndarray:
    """Mean-pools token embeddings for each sentence in the corpus."""
    vocab, vectors = _get_vocab_and_vectors(embedding_model)
    result = []
    for sent in tokenized_sentences:
        valid = [vectors[tok] for tok in sent if tok in vocab]
        result.append(np.mean(valid, axis=0) if valid else np.zeros(vector_size))
    return np.array(result)


def vectorize_document_tfidf(tokens, vocab, vectors, tfidf_row, tfidf_vocab, vector_size=100):
    """Vectorizes a single document using TF-IDF weighted mean pooling."""
    weighted_vectors = []
    weights = []
    for tok in tokens:
        if tok in vocab and tok in tfidf_vocab:
            weight = tfidf_row[0, tfidf_vocab[tok]]
            if weight > 0:
                weighted_vectors.append(vectors[tok] * weight)
                weights.append(weight)
    if not weighted_vectors:
        return np.zeros(vector_size)
    return np.sum(weighted_vectors, axis=0) / np.sum(weights)


def vectorize_corpus_tfidf(tokenized_sentences, embedding_model, tfidf_matrix, tfidf_vectorizer, vector_size=100):
    """Vectorizes a full corpus using TF-IDF weighted mean pooling."""
    vocab, vectors = _get_vocab_and_vectors(embedding_model)
    tfidf_vocab = tfidf_vectorizer.vocabulary_
    return np.array([
        vectorize_document_tfidf(
            tokens=sent,
            vocab=vocab,
            vectors=vectors,
            tfidf_row=tfidf_matrix[i],
            tfidf_vocab=tfidf_vocab,
            vector_size=vector_size
        )
        for i, sent in enumerate(tokenized_sentences)
    ])


def train_word2vec(tokenized_sentences: list, vector_size: int = 100, window: int = 5, min_count: int = 2, sg: int = 0) -> Word2Vec:
    """Trains a Word2Vec model on the given tokenized corpus."""
    model = Word2Vec(
        sentences=tokenized_sentences,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        sg=sg,
        seed=SEED,
        workers=4,
    )
    log_success(f"Word2Vec trained — vocab size: {len(model.wv.key_to_index)}")
    return model


def load_glove_twitter(dim: int = 100):
    """Loads GloVe-Twitter embeddings via gensim downloader (downloads ~400 MB if not cached)."""
    log_info(f"Loading glove-twitter-{dim} (downloads if not cached) ...")
    try:
        model = api.load(f"glove-twitter-{dim}")
        log_success(f"GloVe-Twitter-{dim} loaded — vocab size: {len(model.key_to_index)}")
        return model
    except Exception as e:
        log_error(f"Failed to load GloVe-Twitter-{dim}: {e}")
        raise


def print_embedding_diagnostics(name, X_train, X_val):
    """Prints shape and norm diagnostics required for TM-020."""
    print_separator()
    log_info(f"DIAGNOSTICS: {name}")
    print_separator()
    log_info(f"Train shape: {X_train.shape}")
    log_info(f"Val shape:   {X_val.shape}")
    log_info(f"Train norm:  {np.linalg.norm(X_train):.4f}")
    log_info(f"Val norm:    {np.linalg.norm(X_val):.4f}")
    log_info(f"NaNs:        {np.isnan(X_train).sum() + np.isnan(X_val).sum()}")


def train_and_evaluate_classifier(X_train, X_val, y_train, y_val, classifier, model_name, feature_desc, params):
    """Trains a classifier and logs evaluation metrics."""
    print_separator()
    log_info(f"Training {model_name} on {feature_desc}")
    print_separator()
    classifier.fit(X_train, y_train)
    y_pred = classifier.predict(X_val)
    evaluate_and_log(
        y_val,
        y_pred,
        model_name=model_name,
        feature_desc=feature_desc,
        params=params,
        owner="Bento",
    )


def main():
    print_header("RUNNING WORD EMBEDDINGS COMPARISON PIPELINE")

    # 1. Load Data
    if not os.path.exists(TRAIN_CSV_PATH):
        log_error(f"Training CSV not found at {TRAIN_CSV_PATH}")
        sys.exit(1)

    train_df = pd.read_csv(TRAIN_CSV_PATH)

    # 2. Train/Val Split
    log_info("Creating stratified train/validation split...")
    X_train_raw, X_val_raw, y_train, y_val = stratified_split(train_df)

    # 3. Preprocess Texts to Token Lists
    log_info("Preprocessing tweets into token lists (lemmatizer active)...")
    X_train_tokens = X_train_raw.apply(
        lambda t: preprocess_tweet(t, return_str=False)
    ).tolist()

    X_val_tokens = X_val_raw.apply(
        lambda t: preprocess_tweet(t, return_str=False)
    ).tolist()

    # TF-IDF uses strings, so we join the preprocessed tokens
    X_train_str = [" ".join(tokens) for tokens in X_train_tokens]
    X_val_str = [" ".join(tokens) for tokens in X_val_tokens]

    log_info("Fitting TF-IDF vectorizer for weighted pooling...")
    tfidf_vectorizer = TfidfVectorizer()
    X_train_tfidf_matrix = tfidf_vectorizer.fit_transform(X_train_str)
    X_val_tfidf_matrix = tfidf_vectorizer.transform(X_val_str)

    # 4. Train Word2Vec models: CBOW vs Skip-Gram, 100 vs 200 dimensions
    configs = [
        {"name": "CBOW_100", "vector_size": 100, "sg": 0},
        {"name": "SKIPGRAM_100", "vector_size": 100, "sg": 1},
        {"name": "CBOW_200", "vector_size": 200, "sg": 0},
        {"name": "SKIPGRAM_200", "vector_size": 200, "sg": 1},
    ]

    models = {}

    for config in configs:
        print_separator()
        log_info(f"Training {config['name']} Word2Vec model...")
        print_separator()

        model = train_word2vec(
            tokenized_sentences=X_train_tokens,
            vector_size=config["vector_size"],
            window=5,
            min_count=2,
            sg=config["sg"]
        )

        models[config["name"]] = model

    # Select one 100-dimensional custom Word2Vec model for downstream comparison with GloVe-100
    w2v_model = models["CBOW_100"]

    # 5. Evaluate embeddings qualitatively with similar words
    sanity_words = ["good", "bad", "love", "hate", "movie", "market", "stock"]

    similarity_results = []

    for model_name, model in models.items():
        print_separator()
        log_info(f"SIMILAR WORDS - {model_name}")
        print_separator()

        for word in sanity_words:
            if word not in model.wv.key_to_index:
                log_info(f"{word}: OOV")
                continue

            log_info(f"Most similar to '{word}':")

            for similar_word, score in model.wv.most_similar(word, topn=10):
                print(f"{similar_word:<15} {score:.4f}")

                similarity_results.append({
                    "model": model_name,
                    "query_word": word,
                    "similar_word": similar_word,
                    "similarity": score
                })

    os.makedirs("outputs", exist_ok=True)

    pd.DataFrame(similarity_results).to_csv(
        "outputs/word2vec_similarity.csv",
        index=False
    )
    log_success("Saved Word2Vec similarity results to outputs/word2vec_similarity.csv")

    # 6. Load pre-trained GloVe Twitter Embeddings
    log_info("Loading pre-trained GloVe Twitter 100-dimensional embeddings...")
    glove_model = load_glove_twitter(dim=100)

    # 7. Compute Out-of-Vocabulary (OOV) Statistics on Validation Fold
    log_info("Computing Out-of-Vocabulary (OOV) statistics on validation set...")

    w2v_uniq_oov, w2v_tok_oov, w2v_oov_words = calculate_oov(X_val_tokens, w2v_model)
    glove_uniq_oov, glove_tok_oov, glove_oov_words = calculate_oov(X_val_tokens, glove_model)

    print_separator()
    log_info("OUT-OF-VOCABULARY (OOV) COMPARISON:")
    print_separator()
    log_info(f"Custom Word2Vec Unique Word OOV Rate:   {w2v_uniq_oov * 100:.2f}%")
    log_info(f"Custom Word2Vec Token OOV Rate:         {w2v_tok_oov * 100:.2f}%")
    log_info(f"GloVe-Twitter-100 Unique Word OOV Rate: {glove_uniq_oov * 100:.2f}%")
    log_info(f"GloVe-Twitter-100 Token OOV Rate:       {glove_tok_oov * 100:.2f}%")
    print_separator()

    log_info("Sample OOV Words in Word2Vec:")
    print(list(w2v_oov_words)[:15])

    log_info("Sample OOV Words in GloVe-Twitter-100:")
    print(list(glove_oov_words)[:15])
    print_separator()

    pd.DataFrame([
        {
            "model": "Custom Word2Vec CBOW_100",
            "unique_oov_rate": w2v_uniq_oov,
            "token_oov_rate": w2v_tok_oov,
            "sample_oov_words": list(w2v_oov_words)[:15]
        },
        {
            "model": "GloVe-Twitter-100",
            "unique_oov_rate": glove_uniq_oov,
            "token_oov_rate": glove_tok_oov,
            "sample_oov_words": list(glove_oov_words)[:15]
        }
    ]).to_csv("outputs/oov_comparison.csv", index=False)
    log_success("Saved OOV comparison statistics to outputs/oov_comparison.csv")

    # 8. Vectorize Dataset via Mean Pooling and TF-IDF Weighted Pooling
    log_info("Creating mean pooled vector representations...")

    X_train_w2v_mean = vectorize_corpus(X_train_tokens, w2v_model, 100)
    X_val_w2v_mean = vectorize_corpus(X_val_tokens, w2v_model, 100)

    X_train_glove_mean = vectorize_corpus(X_train_tokens, glove_model, 100)
    X_val_glove_mean = vectorize_corpus(X_val_tokens, glove_model, 100)

    log_info("Creating TF-IDF weighted pooled vector representations...")

    X_train_w2v_tfidf = vectorize_corpus_tfidf(
        X_train_tokens,
        w2v_model,
        X_train_tfidf_matrix,
        tfidf_vectorizer,
        100
    )

    X_val_w2v_tfidf = vectorize_corpus_tfidf(
        X_val_tokens,
        w2v_model,
        X_val_tfidf_matrix,
        tfidf_vectorizer,
        100
    )

    X_train_glove_tfidf = vectorize_corpus_tfidf(
        X_train_tokens,
        glove_model,
        X_train_tfidf_matrix,
        tfidf_vectorizer,
        100
    )

    X_val_glove_tfidf = vectorize_corpus_tfidf(
        X_val_tokens,
        glove_model,
        X_val_tfidf_matrix,
        tfidf_vectorizer,
        100
    )

    embedding_sets = [
        ("Word2Vec Mean Pooling", X_train_w2v_mean, X_val_w2v_mean),
        ("Word2Vec TF-IDF Weighted Pooling", X_train_w2v_tfidf, X_val_w2v_tfidf),
        ("GloVe-Twitter-100 Mean Pooling", X_train_glove_mean, X_val_glove_mean),
        ("GloVe-Twitter-100 TF-IDF Weighted Pooling", X_train_glove_tfidf, X_val_glove_tfidf),
    ]

    print_header("TM-020 DIAGNOSTICS: SHAPES AND NORMS")

    for name, X_train_emb, X_val_emb in embedding_sets:
        print_embedding_diagnostics(name, X_train_emb, X_val_emb)

    log_info("Small sample validation:")
    log_info(f"Original tweet: {X_train_raw.iloc[0]}")
    log_info(f"Tokens: {X_train_tokens[0]}")
    log_info(f"Word2Vec Mean first 5 dims: {X_train_w2v_mean[0][:5]}")
    log_info(f"Word2Vec TF-IDF first 5 dims: {X_train_w2v_tfidf[0][:5]}")

    # 9. Train and Evaluate Downstream Classifiers
    print_header("TM-021 CLASSIFIER EVALUATION")

    for feature_desc, X_train_emb, X_val_emb in embedding_sets:

        train_and_evaluate_classifier(
            X_train=X_train_emb,
            X_val=X_val_emb,
            y_train=y_train,
            y_val=y_val,
            classifier=LogisticRegression(
                penalty='l2',
                solver='lbfgs',
                max_iter=1000,
                class_weight='balanced',
                random_state=SEED
            ),
            model_name="Logistic Regression",
            feature_desc=feature_desc,
            params="penalty=l2, solver=lbfgs, max_iter=1000, class_weight=balanced"
        )

        train_and_evaluate_classifier(
            X_train=X_train_emb,
            X_val=X_val_emb,
            y_train=y_train,
            y_val=y_val,
            classifier=MLPClassifier(
                hidden_layer_sizes=(128,),
                activation='relu',
                solver='adam',
                max_iter=300,
                random_state=SEED
            ),
            model_name="MLP",
            feature_desc=feature_desc,
            params="hidden_layer_sizes=(128,), activation=relu, solver=adam, max_iter=300"
        )

    log_success("WORD EMBEDDINGS COMPARISON PIPELINE COMPLETE!")




### Module: `sentence_embeddings.py`

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer





def get_best_checkpoint(model_name, checkpoint_dir):
    """Returns the path to the fine-tuned checkpoint if it exists, otherwise falls back to base model."""
    if os.path.exists(checkpoint_dir):
        checkpoints = [os.path.join(checkpoint_dir, d) for d in os.listdir(checkpoint_dir) if d.startswith("checkpoint")]
        if checkpoints:
            best_ckpt = sorted(checkpoints, key=lambda x: int(x.split("-")[-1]))[-1]
            log_info(f"Found fine-tuned checkpoint: {best_ckpt}")
            return best_ckpt
    log_info(f"Fine-tuned checkpoint not found locally. Using base model: {model_name}")
    return model_name

def extract_embeddings(texts, model_path, pooling="cls", batch_size=32):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    log_info(f"Loading {model_path} on {device}...")
    
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModel.from_pretrained(model_path)
    model.to(device)
    model.eval()

    all_embs = []
    
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc=f"Extracting {pooling} from {model_path}"):
            batch = texts[i:i+batch_size]
            encoded = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors="pt")
            encoded = {k: v.to(device) for k, v in encoded.items()}
            
            out = model(**encoded)
            hidden = out.last_hidden_state  # (B, L, H)
            
            if pooling == "cls":
                emb = hidden[:, 0, :]
            elif pooling == "mean":
                mask = encoded["attention_mask"].unsqueeze(-1).expand(hidden.size()).float()
                sum_emb = torch.sum(hidden * mask, 1)
                sum_mask = torch.clamp(mask.sum(1), min=1e-9)
                emb = sum_emb / sum_mask
            else:
                raise ValueError("Pooling must be cls or mean")
                
            all_embs.append(emb.cpu().numpy())
            
    return np.vstack(all_embs)

def encoder_features(texts, pooling="cls", batch_size=32, model_path=None):
    """Sentence embeddings from a frozen (not fine-tuned) encoder, as a feature matrix."""
    if model_path is None:
        model_path = DISTILBERT_MODEL_NAME
    return extract_embeddings(list(texts), model_path, pooling=pooling, batch_size=batch_size)

def generate_and_save_embeddings():
    log_info("Loading train data...")
    df = pd.read_csv(TRAIN_CSV_PATH)
    X_train_raw, X_val_raw, y_train, y_val = stratified_split(df)
    
    train_texts = X_train_raw.tolist()
    val_texts = X_val_raw.tolist()
    
    out_dir = "outputs/embeddings"
    os.makedirs(out_dir, exist_ok=True)
    
    models_to_test = {
        "distilbert": get_best_checkpoint(DISTILBERT_MODEL_NAME, DISTILBERT_CHECKPOINT_DIR),
        "finbert": get_best_checkpoint(FINBERT_MODEL_NAME, FINBERT_CHECKPOINT_DIR)
    }
    
    for name, path in models_to_test.items():
        for pooling in ["cls", "mean"]:
            log_info(f"Processing {name} with {pooling} pooling...")
            
            train_path = os.path.join(out_dir, f"X_train_{name}_{pooling}.npy")
            val_path = os.path.join(out_dir, f"X_val_{name}_{pooling}.npy")
            
            if os.path.exists(train_path) and os.path.exists(val_path):
                log_info(f"Embeddings already exist for {name} ({pooling}). Skipping.")
                continue
                
            X_tr = extract_embeddings(train_texts, path, pooling=pooling)
            np.save(train_path, X_tr)
            
            X_va = extract_embeddings(val_texts, path, pooling=pooling)
            np.save(val_path, X_va)
            
    # Save labels
    np.save(os.path.join(out_dir, "y_train.npy"), y_train.values)
    np.save(os.path.join(out_dir, "y_val.npy"), y_val.values)
    log_success(f"All embeddings saved to {out_dir}")



### Module: `experiment.py`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression





def run_tfidf_pipeline(
    X_train_preprocessed, X_val_preprocessed, y_train, y_val,
    ngram_range=(1, 1), min_df=1, max_features=None,
    feature_desc="TF-IDF",
) -> tuple[int, dict]:
    """Fits TF-IDF + LR L2, evaluates on val, logs idempotently. Returns (vocab_size, metrics)."""
    params_str = f"ngram_range={ngram_range}, min_df={min_df}, max_features={max_features}"

    vec = TfidfVectorizer(ngram_range=ngram_range, min_df=min_df, max_features=max_features)
    X_train_vec = vec.fit_transform(X_train_preprocessed)
    X_val_vec = vec.transform(X_val_preprocessed)

    lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)
    lr.fit(X_train_vec, y_train)

    metrics = evaluate_and_log(
        y_val, lr.predict(X_val_vec),
        model_name="Logistic Regression Baseline",
        feature_desc=feature_desc,
        params=params_str,
    )
    return len(vec.vocabulary_), metrics


def run_model_pipeline(
    X_train_vec, X_val_vec, y_train, y_val,
    model, model_name: str, feature_desc: str, params_str: str,
) -> dict:
    """Fits a classifier on pre-vectorized features, evaluates on val, logs idempotently."""
    model.fit(X_train_vec, y_train)
    return evaluate_and_log(
        y_val, model.predict(X_val_vec),
        model_name=model_name,
        feature_desc=feature_desc,
        params=params_str,
    )


def run_classifier(model, X_train_vec, X_val_vec, y_train, y_val,
                   model_name: str, feature_desc: str, params_str: str) -> dict:
    return run_model_pipeline(X_train_vec, X_val_vec, y_train, y_val,
                              model, model_name, feature_desc, params_str)


def run_majority_baseline(y_train, y_val) -> dict:
    """Evaluates the majority-class baseline on the validation set and logs metrics."""
    majority_class = int(pd.Series(y_train).value_counts().idxmax())
    y_pred = np.full(len(y_val), majority_class)
    return evaluate_and_log(
        y_val, y_pred,
        model_name="Majority Class Baseline",
        feature_desc="N/A",
        params=f"majority_class={majority_class} ({LABEL_NAMES[majority_class]})",
    )



### Module: `transformer_trainer.py`

In [ ]:
"""Shared HuggingFace fine-tuning pipeline for sequence classification.

All four encoders (DistilBERT, FinBERT, Twitter-RoBERTa, DeBERTa-v3) share an
identical training loop and differ only in a handful of constants, captured in a
`TrainerSpec` and registered in `SPECS`. Pick a backbone by key:
`run_trainer("finbert", n_samples=1000)`.
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)






MAX_LENGTH = 128


@dataclass(frozen=True)
class TrainerSpec:
    """Everything that distinguishes one encoder trainer from another."""

    display_name: str   # label used in the results leaderboard
    model_name: str     # HF hub id
    cache_dir: str      # tokenized-dataset cache root
    checkpoint_dir: str  # Trainer output root


# All four encoders differ only in config, so they live here as a registry.
# Add a new backbone by adding one entry.
SPECS: dict[str, TrainerSpec] = {
    "distilbert": TrainerSpec("DistilBERT", DISTILBERT_MODEL_NAME,
                              DISTILBERT_CACHE_DIR, DISTILBERT_CHECKPOINT_DIR),
    "finbert":    TrainerSpec("FinBERT", FINBERT_MODEL_NAME,
                              FINBERT_CACHE_DIR, FINBERT_CHECKPOINT_DIR),
    "roberta":    TrainerSpec("Twitter-RoBERTa", ROBERTA_MODEL_NAME,
                              ROBERTA_CACHE_DIR, ROBERTA_CHECKPOINT_DIR),
    "deberta":    TrainerSpec("deberta-v3-base", DEBERTA_MODEL_NAME,
                              DEBERTA_CACHE_DIR, DEBERTA_CHECKPOINT_DIR),
}


def load_tokenizer(spec: TrainerSpec) -> AutoTokenizer:
    log_info(f"Loading tokenizer: {spec.model_name}")
    tokenizer = AutoTokenizer.from_pretrained(spec.model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer


def _tokenize_dataset(tokenizer, texts, labels=None, max_length: int = MAX_LENGTH) -> Dataset:
    """Build a tokenized HF Dataset from texts (and optional labels)."""
    data = {"text": list(texts)}
    if labels is not None:
        data["label"] = list(labels)
    return Dataset.from_dict(data).map(
        lambda batch: tokenizer(batch["text"], truncation=True, max_length=max_length),
        batched=True,
        remove_columns=["text"],
    )


def build_hf_datasets(tokenizer, X_train, X_val, y_train, y_val, cache_dir: Path,
                      max_length: int = MAX_LENGTH):
    cache_dir.mkdir(parents=True, exist_ok=True)

    def _build(texts, labels, cache_path):
        if cache_path.exists():
            log_info(f"Loading dataset from cache: {cache_path}")
            return Dataset.load_from_disk(str(cache_path))
        ds = _tokenize_dataset(tokenizer, texts, labels, max_length=max_length)
        ds.save_to_disk(str(cache_path))
        log_info(f"Dataset cached to: {cache_path}")
        return ds

    return (_build(X_train, y_train, cache_dir / "train"),
            _build(X_val,   y_val,   cache_dir / "val"))


def make_compute_metrics(spec: TrainerSpec, notes: str = "", params: str = ""):
    def _compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        metrics = compute_metrics(labels, preds)
        log_model_run(
            model_name=spec.display_name,
            feature_desc="HF fine-tune",
            metrics=metrics,
            params=params or f"model={spec.model_name}, max_length={MAX_LENGTH}",
            notes=notes,
        )
        return {"accuracy": metrics["accuracy"], "f1_macro": metrics["f1_macro"]}
    return _compute_metrics


def build_trainer(spec: TrainerSpec, model, tokenizer, train_ds, val_ds, notes: str = "",
                  learning_rate: float = 2e-5, batch_size: int = 16,
                  seed: int = SEED, params: str = "") -> Trainer:
    training_args = TrainingArguments(
        output_dir=str(Path(spec.checkpoint_dir) / spec.model_name.replace("/", "_")),
        num_train_epochs=3,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        learning_rate=learning_rate,
        weight_decay=0.01,
        warmup_ratio=0.1,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        seed=seed,
        logging_steps=50,
        report_to="none",
    )
    return Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=make_compute_metrics(spec, notes=notes, params=params),
    )


def run_trainer(model: str, n_samples: int | None = 500, notes: str = "",
                learning_rate: float = 2e-5, max_length: int = MAX_LENGTH,
                seed: int = SEED, batch_size: int = 16) -> Trainer:
    """Fine-tune one of the registered encoders.

    `model` is a key of SPECS: "distilbert", "finbert", "roberta", "deberta".
    `learning_rate`, `max_length`, `seed` and `batch_size` allow controlled
    variations; each combination gets its own leaderboard row and dataset cache.
    """
    spec = SPECS[model]

    log_info(f"Loading {'all' if n_samples is None else n_samples} samples ...")
    df = pd.read_csv(TRAIN_CSV_PATH)
    if n_samples:
        df = df.sample(n=n_samples, random_state=SEED).reset_index(drop=True)
    X_train, X_val, y_train, y_val = stratified_split(df)
    log_info(f"Split — train={len(X_train)}, val={len(X_val)}")

    tokenizer = load_tokenizer(spec)

    # Sample-size-specific cache dir so a 500-sample spike isn't served when we
    # ask for the full dataset. max_length variants get their own cache too.
    suffix = "full" if n_samples is None else f"n{n_samples}"
    if max_length != MAX_LENGTH:
        suffix += f"_len{max_length}"
    cache_dir = Path(spec.cache_dir) / suffix
    train_ds, val_ds = build_hf_datasets(tokenizer, X_train, X_val, y_train, y_val, cache_dir,
                                         max_length=max_length)
    log_info(f"Datasets — train={len(train_ds)} rows, val={len(val_ds)} rows")

    log_info(f"Loading {spec.model_name} sequence classifier ...")
    # ignore_mismatched_sizes=True safely re-initializes the classification head
    # for our project's 3-label schema.
    hf_model = AutoModelForSequenceClassification.from_pretrained(
        spec.model_name,
        num_labels=NUM_LABELS,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        ignore_mismatched_sizes=True,
    )

    params = (f"model={spec.model_name}, max_length={max_length}, lr={learning_rate}, "
              f"seed={seed}, n={'full' if n_samples is None else n_samples}")
    trainer = build_trainer(spec, hf_model, tokenizer, train_ds, val_ds, notes=notes,
                            learning_rate=learning_rate, batch_size=batch_size,
                            seed=seed, params=params)

    log_info("Starting training ...")
    trainer.train()
    log_info("Running final evaluation ...")
    trainer.evaluate()
    log_success("Training complete.")
    return trainer


def predict_test_set(trainer, tokenizer, test_csv_path: str = TEST_CSV_PATH) -> np.ndarray:
    log_info(f"Loading test set from {test_csv_path}")
    test_df = pd.read_csv(test_csv_path)
    log_info(f"Test rows: {len(test_df)}")

    test_ds = _tokenize_dataset(tokenizer, test_df["text"])
    preds = trainer.predict(test_ds)
    labels = np.argmax(preds.predictions, axis=-1)
    log_success(f"Generated {len(labels)} test predictions.")
    return labels



### Module: `decoder_qwen.py`

In [ ]:
from __future__ import annotations

import re

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer







LABEL_TO_ID = {v.lower(): k for k, v in LABEL_NAMES.items()}
MAJORITY_FALLBACK = 2  # Neutral — dominant class (~62 % of train set)

SYSTEM_PROMPT = (
    "You are a financial sentiment classifier for tweets about stocks. "
    "Each tweet is one of three classes: Bearish, Bullish, or Neutral. "
    "Reply with exactly ONE word — the class name — and nothing else."
)


# ── Model loading ────────────────────────────────────────────────────────────

def load_qwen(
    device: str | None = None,
    dtype: torch.dtype | None = None,
) -> tuple[AutoTokenizer, AutoModelForCausalLM]:
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    if dtype is None:
        dtype = torch.bfloat16 if device == "cuda" else torch.float32

    log_info(f"Loading tokenizer: {QWEN_MODEL_NAME}")
    tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_NAME)
    # Decoder-only models need left padding so the last token == the prediction position.
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    log_info(f"Loading model on {device} with dtype={dtype} ...")
    model = AutoModelForCausalLM.from_pretrained(
        QWEN_MODEL_NAME,
        torch_dtype=dtype,
        device_map=device,
    )
    model.eval()
    log_success("Qwen ready.")
    return tokenizer, model


# ── Prompt construction ──────────────────────────────────────────────────────

def _sample_few_shot_examples(train_df: pd.DataFrame, n_per_class: int, seed: int) -> list[tuple[str, str]]:
    examples: list[tuple[str, str]] = []
    rng = np.random.RandomState(seed)
    for label_id, label_name in LABEL_NAMES.items():
        pool = train_df.loc[train_df["label"] == label_id, "text"].tolist()
        idxs = rng.choice(len(pool), size=min(n_per_class, len(pool)), replace=False)
        for i in idxs:
            examples.append((pool[i], label_name))
    rng.shuffle(examples)  # avoid grouping by class
    return examples


def build_few_shot_messages(
    train_df: pd.DataFrame,
    n_per_class: int = 2,
    seed: int = SEED,
) -> list[dict]:
    messages: list[dict] = [{"role": "system", "content": SYSTEM_PROMPT}]
    for text, label_name in _sample_few_shot_examples(train_df, n_per_class, seed):
        messages.append({"role": "user", "content": f"Tweet: {text}"})
        messages.append({"role": "assistant", "content": label_name})
    return messages


# ── Inference ────────────────────────────────────────────────────────────────

_LABEL_REGEX = re.compile(r"(bearish|bullish|neutral)", re.IGNORECASE)


def _parse_label(generated_text: str) -> int | None:
    m = _LABEL_REGEX.search(generated_text[:32])
    if m is None:
        return None
    return LABEL_TO_ID[m.group(1).lower()]


def _build_chat_string(tokenizer, few_shot_messages: list[dict], tweet: str) -> str:
    messages = list(few_shot_messages) + [{"role": "user", "content": f"Tweet: {tweet}"}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


@torch.no_grad()
def classify_batch(
    tokenizer,
    model,
    few_shot_messages: list[dict],
    tweets: list[str],
    batch_size: int = 8,
    max_new_tokens: int = 4,
) -> tuple[list[int], int]:
    preds: list[int] = []
    parse_fails = 0
    device = next(model.parameters()).device

    for start in range(0, len(tweets), batch_size):
        chunk = tweets[start:start + batch_size]
        prompts = [_build_chat_string(tokenizer, few_shot_messages, t) for t in chunk]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=2048).to(device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
        new_tokens = outputs[:, inputs["input_ids"].shape[1]:]
        decoded = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)

        for text in decoded:
            pid = _parse_label(text)
            if pid is None:
                parse_fails += 1
                preds.append(MAJORITY_FALLBACK)
            else:
                preds.append(pid)

        if (start // batch_size) % 10 == 0:
            log_info(f"  classified {start + len(chunk)}/{len(tweets)} (fails so far: {parse_fails})")

    return preds, parse_fails


def classify_tweet(tokenizer, model, few_shot_messages: list[dict], tweet: str) -> int:
    preds, _ = classify_batch(tokenizer, model, few_shot_messages, [tweet], batch_size=1)
    return preds[0]


# ── End-to-end evaluation ────────────────────────────────────────────────────

def run_qwen_eval(
    owner: str = "La Feria",
    notes: str = "TM-031 EXTRA WORK +1.0pt",
    n_per_class: int = 2,
    batch_size: int = 8,
    device: str | None = None,
    val_subset: int | None = None,
    tokenizer=None,
    model=None,
) -> dict:
    df = pd.read_csv(TRAIN_CSV_PATH)
    X_train, X_val, y_train, y_val = stratified_split(df)
    train_df = pd.DataFrame({"text": X_train.values, "label": y_train.values})

    if val_subset is not None:
        X_val = X_val.iloc[:val_subset]
        y_val = y_val.iloc[:val_subset]

    if tokenizer is None or model is None:
        tokenizer, model = load_qwen(device=device)
    few_shot_messages = build_few_shot_messages(train_df, n_per_class=n_per_class, seed=SEED)
    n_shots = n_per_class * NUM_LABELS

    log_info(f"Classifying {len(X_val)} val tweets with {n_shots}-shot Qwen ...")
    preds, parse_fails = classify_batch(
        tokenizer, model, few_shot_messages, X_val.tolist(), batch_size=batch_size,
    )
    parse_fail_rate = parse_fails / max(len(preds), 1)
    if parse_fail_rate > 0:
        log_warning(f"Parse-failure rate: {parse_fail_rate:.2%} ({parse_fails}/{len(preds)})")

    metrics = compute_metrics(y_val.values, np.array(preds))

    log_model_run(
        model_name="Qwen2.5-1.5B-Instruct",
        feature_desc=f"few-shot {n_shots}-shot ({n_per_class}/class) chat template",
        metrics=metrics,
        params=f"dtype=bf16, greedy, max_new_tokens=4, seed={SEED}, batch={batch_size}",
        owner=owner,
        notes=f"{notes}; parse_fail_rate={parse_fail_rate:.2%}",
    )
    log_success(f"Qwen run logged. f1_macro={metrics['f1_macro']:.4f}")
    return metrics



### Module: `error_analysis.py`

In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix






def _plot_confusion_matrix(cm, path: str, model_name: str, feature_desc: str) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=["Bearish (0)", "Bullish (1)", "Neutral (2)"],
        yticklabels=["Bearish (0)", "Bullish (1)", "Neutral (2)"],
    )
    plt.title(f"Confusion Matrix\n{model_name} ({feature_desc})")
    plt.xlabel("Predicted Class")
    plt.ylabel("Actual Class")
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()
    log_success(f"Confusion matrix saved to {path}")


def _collect_errors(y_val, y_pred, y_proba, X_val_raw, X_val_pre) -> tuple[pd.DataFrame, dict]:
    errors = []
    for (orig_idx, raw_text), clean_text, actual, predicted, probas in zip(
        X_val_raw.items(), X_val_pre, y_val, y_pred, y_proba
    ):
        if actual == predicted:
            continue
        errors.append({
            "original_index": int(orig_idx),
            "tweet_text": str(raw_text),
            "cleaned_text": str(clean_text),
            "actual_label": int(actual),
            "actual_name": LABEL_MAPPING[actual],
            "predicted_label": int(predicted),
            "predicted_name": LABEL_MAPPING[predicted],
            "confidence": float(probas[predicted]),
            "actual_class_proba": float(probas[actual]),
            "probabilities": {LABEL_MAPPING[i]: float(p) for i, p in enumerate(probas)},
        })

    errors_df = pd.DataFrame(errors)
    top_per_class = {
        LABEL_MAPPING[cls]: (
            errors_df[errors_df["actual_label"] == cls]
            .sort_values("confidence", ascending=False)
            .head(20)
            .to_dict(orient="records")
        )
        for cls in [0, 1, 2]
    }
    return errors_df, top_per_class


def _write_txt_report(
    errors_df: pd.DataFrame, top_per_class: dict, cm, y_val,
    path: str, model_name: str, feature_desc: str,
) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write("=" * 100 + "\n")
        f.write("SENTIMENT CLASSIFICATION ERROR ANALYSIS REPORT\n")
        f.write("=" * 100 + "\n\n")
        f.write(f"Model : {model_name} | Features : {feature_desc}\n")
        f.write(f"Val size : {len(y_val)} | Errors : {len(errors_df)} ({len(errors_df)/len(y_val)*100:.2f}%)\n\n")

        f.write("CONFUSION MATRIX:\n")
        for i, name in LABEL_MAPPING.items():
            row = ", ".join(f"{LABEL_MAPPING[j]}={cm[i][j]}" for j in range(3))
            f.write(f"  {name} → {row}\n")
        f.write("\n" + "=" * 100 + "\n")
        f.write("TOP 20 MOST CONFIDENT MISCLASSIFICATIONS BY CLASS\n")
        f.write("=" * 100 + "\n\n")

        for class_name, errs in top_per_class.items():
            f.write(f"{'#' * 40}\nACTUAL CLASS: {class_name.upper()}\n{'#' * 40}\n\n")
            if not errs:
                f.write("No errors in this class.\n\n")
                continue
            for i, err in enumerate(errs, 1):
                f.write(f"{i}. \"{err['tweet_text']}\"\n")
                f.write(f"   Clean  : \"{err['cleaned_text']}\"\n")
                f.write(f"   Actual : {err['actual_name']} (p={err['actual_class_proba']:.4f})"
                        f"  →  Predicted : {err['predicted_name']} (p={err['confidence']:.4f})\n")
                f.write(f"   Probas : {err['probabilities']}\n")
                f.write("-" * 80 + "\n")
            f.write("\n")
    log_success(f"TXT report saved to {path}")


def _write_json_report(top_per_class: dict, path: str) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(top_per_class, f, indent=2)
    log_success(f"JSON report saved to {path}")


def run_error_analysis(
    y_val, y_pred, y_proba,
    X_val_raw: pd.Series, X_val_pre: pd.Series,
    model_name: str = "Logistic Regression",
    feature_desc: str = "TF-IDF",
) -> None:
    """Plots confusion matrix and writes misclassification reports for any fitted model."""
    metrics = compute_metrics(y_val, y_pred)
    print_separator()
    log_info(f"Val F1 Macro : {metrics['f1_macro']:.4f}")
    log_info(f"Val Accuracy : {metrics['accuracy']:.4f}")
    print_separator()

    cm = confusion_matrix(y_val, y_pred)
    _plot_confusion_matrix(cm, CONF_MATRIX_PLOT_PATH, model_name, feature_desc)

    log_info("Collecting misclassifications ...")
    errors_df, top_per_class = _collect_errors(y_val, y_pred, y_proba, X_val_raw, X_val_pre)
    _write_txt_report(errors_df, top_per_class, cm, y_val,
                      MISCLASSIFIED_TXT_PATH, model_name, feature_desc)
    _write_json_report(top_per_class, MISCLASSIFIED_JSON_PATH)



In [ ]:

# Make the project root the working directory so `src/` and the data paths resolve
# whether the notebook is run from notebooks/ or from the repo root.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))



sns.set_theme(style="whitegrid")
print("Working directory:", os.getcwd())

## 1. Why Twitter-RoBERTa

We compared four fine-tuned encoders on the full training set. DistilBERT looked best on small samples, but once every model was trained on all the data Twitter-RoBERTa won. It is pre-trained on 58M tweets, so it already handles the register of our data: cashtags, slang, mentions and short noisy text. FinBERT, pre-trained on formal financial reports, came a close second; the topic matched but the register did not. DeBERTa-v3 collapsed during fine-tuning. Full validation macro F1: Twitter-RoBERTa 0.845, FinBERT 0.828, DistilBERT 0.822. The full comparison is in `tm_tests_31.ipynb`.

## 2. Fine-tune on the full training set

`run_trainer` loads the data, applies the 80/20 stratified split, fine-tunes for 3 epochs and keeps the best epoch by macro F1. `n_samples=None` uses the full training set. Every epoch is logged to `outputs/results.csv`.

In [ ]:
trainer = run_trainer("roberta", n_samples=None, notes="tm_final champion (full data)")

## 3. Validation performance

We score the best checkpoint on the validation split and show the per-class breakdown and the confusion matrix. The Bearish class is the hardest, which matches the error analysis in `tm_tests_31.ipynb`.

In [ ]:
val_out = trainer.predict(trainer.eval_dataset)
y_true = val_out.label_ids
y_pred = np.argmax(val_out.predictions, axis=-1)

m = compute_metrics(y_true, y_pred)
print(f"Accuracy        : {m['accuracy']:.4f}")
print(f"Macro precision : {m['precision_macro']:.4f}")
print(f"Macro recall    : {m['recall_macro']:.4f}")
print(f"Macro F1        : {m['f1_macro']:.4f}")
print("Per-class F1    :", {k: round(v, 4) for k, v in m['f1_per_class'].items()})

ConfusionMatrixDisplay.from_predictions(
    y_true, y_pred, display_labels=list(LABEL_NAMES.values()), cmap="Blues")
plt.title("Twitter-RoBERTa: validation confusion matrix")
plt.tight_layout(); plt.show()

## 4. Test predictions and submission

We classify the held-out test set with the best checkpoint and write the two-column submission (`id`, `label`) to `outputs/pred_31.csv`.

In [ ]:
tokenizer = load_tokenizer(SPECS["roberta"])
preds = predict_test_set(trainer, tokenizer, test_csv_path=TEST_CSV_PATH)

test_df = pd.read_csv(TEST_CSV_PATH)
submission = save_submission(test_df, preds, output_path="outputs/pred_31.csv", id_col="id")

print("
Prediction distribution:")
print(pd.Series(preds).map(LABEL_NAMES).value_counts())
submission.head()